# 05. 딥 리서치 에이전트 — 병렬 서브에이전트와 5단계 워크플로

## 학습 목표

- 병렬 서브에이전트 3개(researcher-1, researcher-2, fact-checker)를 구성한다
- `think_tool`로 전략적 반성(strategic reflection)을 구현한다
- 5단계 워크플로(Plan → Delegate → Synthesize → Verify → Report)를 설계한다
- v1 미들웨어(SummarizationMiddleware, ModelCallLimitMiddleware, ModelFallbackMiddleware)를 적용한다


## 개요

| 항목 | 내용 |
|------|------|
| **프레임워크** | Deep Agents |
| **핵심 컴포넌트** | 병렬 서브에이전트 3개, think_tool |
| **워크플로** | 5단계: Plan → Delegate → Synthesize → Verify → Report |
| **백엔드** | `FilesystemBackend(root_dir=".", virtual_mode=True)` |
| **빌트인 도구** | `write_todos` (계획), `task` (서브에이전트 호출) |
| **스킬** | `skills/deep-research/SKILL.md` — 리서치 방법론 + 인용 규칙 |

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY를 .env에 설정하세요"
assert os.environ.get("TAVILY_API_KEY"), "TAVILY_API_KEY를 .env에 설정하세요"

In [3]:
# Observability 설정 (선택)
import os

if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault(
        "LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default")
    )
    print(f"LangSmith tracing ON — project: {os.environ['LANGCHAIN_PROJECT']}")

LangSmith tracing ON — project: LangGraph-Tutorial


In [4]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-5.4-mini")

/home/sds/projects/SDS-AX-Advanced-2026-1/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1단계: think_tool — 전략적 반성 도구

`think_tool`은 에이전트가 행동하기 전에 "생각"을 기록하는 도구입니다. 이 패턴은 에이전트의 의사결정 품질을 높입니다:

- 검색 결과를 분석하고 다음 행동을 계획
- 수집된 정보의 충분성을 평가
- 서브에이전트에게 위임할 작업을 구체화


In [5]:
from langchain.tools import tool


@tool
def think_tool(thought: str) -> str:
    """전략적 반성 — 현재 상황을 분석하고 다음 행동을 계획합니다."""
    return f"Reflection recorded: {thought}"


## 2단계: web_search 도구 (Tavily)

Tavily API를 사용하여 실제 웹 검색을 수행하는 도구를 정의합니다.

In [6]:
from typing import Literal
from tavily import TavilyClient

tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])


@tool
def web_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
) -> str:
    """웹 검색을 수행합니다 (Tavily API).

    Args:
        query: 검색할 질문 또는 키워드
        max_results: 반환할 최대 결과 수
        topic: 검색 주제 카테고리
    """
    results = tavily_client.search(query, max_results=max_results, topic=topic)
    # 결과를 읽기 쉬운 문자열로 변환
    output_parts = []
    for r in results.get("results", []):
        output_parts.append(f"- [{r['title']}]({r['url']})\n  {r['content']}")
    return "\n\n".join(output_parts) if output_parts else f"'{query}'에 대한 검색 결과를 찾을 수 없습니다."

## 3단계: 5단계 리서치 워크플로 프롬프트

의 가 프롬프트를 로드합니다 (LangSmith Hub -> Langfuse -> 기본값).

| 단계 | 이름 | 설명 |
|------|------|------|
| 1 | **Plan** | 로 리서치 계획 작성 |
| 2 | **Delegate** | 서브에이전트에게 병렬 조사 위임 (최대 3개 동시) |
| 3 | **Synthesize** | 수집된 정보를 통합 |
| 4 | **Verify** | fact-checker가 사실 검증 |
| 5 | **Report** | 최종 보고서 작성 |

In [7]:
from prompts import RESEARCH_AGENT_PROMPT

print(RESEARCH_AGENT_PROMPT)

당신은 박사급 딥 리서치 에이전트입니다.

## 워크플로
1. **Plan**: write_todos로 리서치 계획을 세우세요
2. **Delegate**: 서브에이전트에게 조사를 위임하세요 (비교 분석 시 병렬)
3. **Synthesize**: 수집된 정보를 통합하세요
4. **Verify**: fact-checker에게 사실 검증을 요청하세요
5. **Report**: 최종 보고서를 작성하고 파일로 저장하세요

## 규칙
- 검색 후 반드시 think_tool로 반성하세요
- 서브에이전트는 최대 3개까지 병렬 실행
- 인용은 [1], [2] 형식으로, 출처 섹션을 포함하세요
- 단순 주제는 서브에이전트 1개, 비교 분석은 2-3개 사용하세요
- 최종 보고서는 반드시 edit_file 도구로 /reports/ 폴더에 마크다운 파일로 저장하세요
- 파일명은 주제를 요약한 영문 snake_case (예: /reports/ai_agent_frameworks.md)


## 4단계: 서브에이전트 3개 정의

딥 리서치 에이전트는 3개의 전문 서브에이전트를 사용합니다:

| 서브에이전트 | 역할 | 도구 |
|------------|------|------|
| `researcher-1` | 주제 조사 담당 | web_search, think_tool |
| `researcher-2` | 비교/보완 조사 | web_search, think_tool |
| `fact-checker` | 사실 검증 담당 | web_search |


In [8]:
researcher_1 = {
    "name": "researcher-1",
    "description": "주제에 대한 심층 조사를 수행합니다",
    "system_prompt": "당신은 리서치 전문가입니다. 주제를 깊이 조사하고 핵심 정보를 요약하세요. 검색 후 think_tool로 반성하세요.",
    "tools": [web_search, think_tool],
}


In [9]:
researcher_2 = {
    "name": "researcher-2",
    "description": "보완적 관점에서 추가 조사를 수행합니다",
    "system_prompt": "당신은 보완 리서처입니다. 다른 관점에서 추가 정보를 수집하세요. 검색 후 think_tool로 반성하세요.",
    "tools": [web_search, think_tool],
}


In [10]:
fact_checker = {
    "name": "fact-checker",
    "description": "수집된 정보의 사실 여부를 검증합니다",
    "system_prompt": "당신은 팩트체커입니다. 제공된 정보의 정확성을 검증하고, 오류가 있으면 지적하세요.",
    "tools": [web_search],
}


## 5단계: 딥 리서치 에이전트 생성 (v1 미들웨어 + CompositeBackend)

모든 도구와 서브에이전트를 조합하여 최종 에이전트를 생성합니다.

### CompositeBackend 라우팅 전략
06_memory_and_skills에서 배운 `CompositeBackend`를 활용하여,
`/reports/` 경로만 실제 디스크에 저장하고 나머지는 가상 모드로 유지합니다.

| 경로 | 백엔드 | 저장 위치 |
|------|--------|---------|
| `/reports/*` | `FilesystemBackend(virtual_mode=True)` | 실제 디스크 (`research_outputs/`) |
| 그 외 | `StateBackend` (기본) | state 메모리 (안전) |

> **주의**: CompositeBackend는 라우트 prefix(`/reports/`)를 strip한 후 `/{suffix}` 형태로 전달합니다.
> `virtual_mode=True`여야 이 경로가 `root_dir` 기준으로 해석됩니다.
> `virtual_mode=False`이면 절대 경로(`/xxx.md`)로 인식하여 root_dir을 무시합니다.

### v1 미들웨어

| 미들웨어 | 역할 |
|---------|------|
| `SummarizationMiddleware` | 긴 리서치 대화를 자동 요약하여 컨텍스트 절약 |
| `ModelCallLimitMiddleware` | 리서치 무한 루프 방지 — 최대 30회 모델 호출 제한 |
| `ModelFallbackMiddleware` | 주 모델 실패 시 백업 모델로 자동 전환 |

In [11]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend, StateBackend, CompositeBackend
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import (
    SummarizationMiddleware,
    ModelCallLimitMiddleware,
    ModelFallbackMiddleware,
)
from pathlib import Path

# /reports/ 출력 디렉토리 생성
reports_dir = Path("research_outputs")
reports_dir.mkdir(exist_ok=True)


# CompositeBackend: /reports/만 실제 디스크, 나머지는 가상(안전)
# CompositeBackend는 라우트 prefix를 strip 후 "/suffix" 형태로 전달하므로
# FilesystemBackend에 virtual_mode=True를 설정해야 root_dir 기준으로 경로 해석됨
# (virtual_mode=False이면 "/xxx.md"를 절대 경로로 인식 → root_dir 무시 → 오류)
def research_backend_factory(runtime):
    return CompositeBackend(
        default=StateBackend(runtime),
        routes={
            "/reports/": FilesystemBackend(
                root_dir=str(reports_dir),
                virtual_mode=True,
            ),
        },
    )


research_agent = create_deep_agent(
    model=model,
    tools=[web_search, think_tool],
    subagents=[researcher_1, researcher_2, fact_checker],
    system_prompt=RESEARCH_AGENT_PROMPT,
    backend=research_backend_factory,
    skills=["/skills/"],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(model=model, trigger=("messages", 15)),
        ModelCallLimitMiddleware(run_limit=30),
        ModelFallbackMiddleware("gpt-4.1-mini"),
    ],
)

## 6단계: 리서치 실행

에이전트에게 리서치 주제를 부여하면 5단계 워크플로를 자동으로 수행합니다.


In [12]:
thread = {"configurable": {"thread_id": "research-1"}}
response = research_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "AI 에이전트 프레임워크의 현재 동향을 조사해주세요. LangGraph와 Deep Agents를 중심으로 비교 분석해주세요.",
            }
        ]
    },
    config={**thread, **{}},
)
print(response["messages"][-1].content)

조사 완료했습니다. 보고서는 파일로 저장했습니다:

`/reports/ai_agent_frameworks_langgraph_vs_deep_agents.md`

핵심 결론만 요약하면:
- **LangGraph**: 상태 기반 그래프 오케스트레이션, checkpointing, interrupt, durable execution 중심의 **프로덕션용 제어층**
- **Deep Agents**: 장기 작업, 서브에이전트, 파일시스템/메모리/스킬 기반의 **agent harness**
- 시장 동향은 둘 다 공통적으로 **자율성보다 통제 가능성, 재현성, 운영성** 쪽으로 이동 중입니다.

원하시면 다음 단계로
1. **표 한 장짜리 의사결정 매트릭스**
2. **LangGraph 도입 체크리스트**
3. **임원 보고용 1페이지 요약본**
으로 재정리해드릴 수 있습니다.


## 6-1단계: 연구 결과 파일 확인

`CompositeBackend`의 `/reports/` 라우팅에 의해, 에이전트가 `edit_file("/reports/xxx.md", ...)`을
호출하면 `research_outputs/` 디렉토리에 **직접 디스크 파일이 생성**됩니다.

별도의 추출 코드 없이 에이전트 실행만으로 파일이 생깁니다.

In [13]:
from pathlib import Path

# CompositeBackend가 /reports/ 경로를 디스크에 직접 저장했는지 확인
print("=== research_outputs/ 디렉토리 확인 ===")
for f in sorted(reports_dir.glob("*.md")):
    print(f"  📄 {f.name} ({f.stat().st_size:,} bytes)")
    # 첫 5줄 미리보기
    lines = f.read_text(encoding="utf-8").splitlines()[:5]
    for line in lines:
        print(f"     {line}")
    print()

=== research_outputs/ 디렉토리 확인 ===
  📄 ai_agent_frameworks_langgraph_vs_deep_agents.md (6,878 bytes)
     # AI 에이전트 프레임워크 동향: LangGraph vs Deep Agents
     
     ## 요약
     AI 에이전트 프레임워크의 최근 흐름은 **자율성 극대화**보다 **제어 가능성, 지속 실행, 관측성, 인간 개입, 재현성**으로 이동하고 있다. 이 흐름에서 **LangGraph**는 상태 기반 그래프 오케스트레이션과 프로덕션 운영 기능으로, **Deep Agents**는 장기 작업·서브에이전트·파일시스템 중심의 agent harness로 각각 다른 위치를 차지한다. 둘은 경쟁 관계이기도 하지만, 실제로는 **런타임( LangGraph ) vs 작업 실행 하니스(Deep Agents)**로 보아야 정확하다 [1][2].
     



## 7단계: 스트리밍 — 네임스페이스 추적

`stream(subgraphs=True)`로 메인 에이전트와 서브에이전트의 실행 과정을 네임스페이스별로 추적합니다. 어떤 서브에이전트가 언제 호출되는지 실시간으로 확인할 수 있습니다.


In [14]:
# 네임스페이스별 청크 수 확인
from collections import Counter

thread2 = {"configurable": {"thread_id": "research-2"}}
chunks = []
for ns, chunk in research_agent.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "Deep Agents의 서브에이전트 아키텍처에 대해 조사해주세요.",
            }
        ]
    },
    config={**thread2, **{}},
    subgraphs=True,
):
    chunks.append((ns, chunk))

ns_counts = Counter(ns for ns, _ in chunks)
print(f"총 {len(chunks)}개 청크 수신", flush=True)
for ns_name, cnt in ns_counts.items():
    label = ns_name if ns_name else "main"
    print(f"  [{label}] {cnt}개 청크", flush=True)

총 97개 청크 수신
  [main] 50개 청크
  [('tools:97f3aa4a-1f88-6d03-bcab-f23ba97becec',)] 20개 청크
  [('tools:e681db87-d575-be95-2271-837cacc6a4ec',)] 16개 청크
  [('tools:bc34dfa5-fd3a-71f2-2901-ddc3f60a7d97',)] 11개 청크


## 서브에이전트 설계 모범 사례

| 원칙 | 설명 |
|------|------|
| **명확한 설명** | `description`을 구체적으로 작성 — 메인 에이전트가 위임 대상을 선택하는 기준 |
| **전문 프롬프트** | `system_prompt`에 출력 형식, 제약, 워크플로 포함 |
| **최소 도구** | 필요한 도구만 할당 — 불필요한 도구는 혼란 유발 |
| **간결한 결과** | 서브에이전트가 요약을 반환하도록 지시 — 원시 데이터 전달 금지 |


## 요약

| 항목 | 핵심 |
|------|------|
| **think_tool** | 전략적 반성 — 검색 후 분석, 다음 행동 계획 |
| **서브에이전트** | researcher-1, researcher-2, fact-checker 병렬 실행 |
| **워크플로** | Plan → Delegate → Synthesize → Verify → Report |
| **컨텍스트 관리** | 서브에이전트 결과만 메인에 전달 — 중간 과정 격리 |